# A2 E0 — Phát hiện biển số với YOLOv5s

Notebook hoàn chỉnh cho pipeline **detection-only**: kiểm kê tài nguyên, kiểm tra ảnh/nhãn YOLO, cấu hình official YOLOv5s, fine-tune, xác minh checkpoint, đánh giá validation/test, phân tích theo subset, trực quan hóa và xuất kết quả.

## Phạm vi và yêu cầu A2

**Yêu cầu trong brief A2 (Assignment 2, CO5085, mục 1.1, 2, 3, 4.1, 4.1.2, 4.2, 5–7):** cần pipeline huấn luyện/fine-tune và đánh giá; tập công khai đủ lớn, augmentation, metric đúng bài toán; so sánh ít nhất hai phương pháp/biến thể trong toàn dự án; detection dùng mAP@0.5 (có thể thêm mAP@0.5:0.95/FPS); phân tích paper, thực nghiệm, hạn chế và liên hệ; nộp code, PDF, GitHub Pages, video demo.

**Lựa chọn triển khai trong notebook này:** chỉ E0 detection bằng YOLOv5s official repository (khớp detector của paper; nếu đề xuất A2.1 ghi YOLO26n thì phải nêu lý do đổi trong báo cáo); giữ split train/val/test hiện có; YOLO labels một lớp `license_plate`; dùng validation để theo dõi/chọn checkpoint; test chỉ chạy đánh giá cuối, không dùng để tuning. E1 CRNN và E2 PDLPR-inspired recognition ở ngoài phạm vi. Vì vậy E0 tự nó không đáp ứng yêu cầu so sánh ≥2 phương pháp/biến thể của A2; phép so sánh phải được thực hiện ở phạm vi dự án (ví dụ E1/E2) hoặc bổ sung detector variant có kiểm soát.

**Paper được cung cấp:** Tao et al., “A Real-Time License Plate Detection and Recognition Model in Unconstrained Scenarios,” *Sensors* 24 (2024), 2791, DOI [10.3390/s24092791](https://doi.org/10.3390/s24092791). Paper dùng YOLOv5 cho detection và đề xuất PDLPR cho recognition; notebook này chỉ dùng YOLOv5s làm baseline detection, không tái hiện PDLPR. Paper báo cáo trung bình recognition accuracy 99.4% và 159.8 FPS trên các thí nghiệm CCPD; không xem đó là kết quả detection hay kết quả của dự án này.

> Không cell nào đồng bộ, sao chép, rebuild, ghi đè hoặc xóa dữ liệu gốc. Đồng bộ dataset được để sau theo yêu cầu. Chưa huấn luyện và chưa có metrics cho tới khi người dùng bật cell tương ứng sau khi xác minh prerequisite.


## Thay đổi so với bản trước (v2)
- **Sửa lỗi:** `Counter`/`defaultdict` được import ở cell môi trường; trước đây cell kiểm kê dùng `Counter` trước khi import nên báo `NameError` khi có `manifest.csv`.
- **Cấu hình:** `EPOCHS` 100 → 50, thêm `PATIENCE` (đang dùng cho `--patience`), thêm `PAPER_IOU=0.7`.
- **Cảnh báo subset:** cell EDA báo nếu `images/test` không chia theo subset, vì khi đó phân tích theo subset chỉ còn nhóm `(root)`.
- **Mới (9b):** ghi lại augmentation và siêu tham số thực tế từ `hyp.yaml`/`opt.yaml` để viết mục Dataset/DataLoader/augmentation.
- **Mới (12b):** độ chính xác theo ảnh với IoU > 0.7 (cùng tiêu chí paper) và FPS batch=1, tách khỏi mAP@0.5.
- **Export:** tự parse P/R/mAP@0.5/mAP@0.5:0.95/speed từ log `val.py` (toàn test, từng subset), gom cả kết quả 12b và augmentation vào `e0_report.json`.

Các cờ `RUN_*` vẫn mặc định `False`: bật từng bước theo thứ tự khi đã sẵn sàng.

## Cách chạy
Chạy lần lượt từ đầu trong Google Colab. Cấp quyền Drive khi được hỏi. Trước tiên sửa duy nhất cell **Cấu hình trung tâm**. Chạy kiểm kê trước khi chọn dataset. Các bước setup/train/evaluate đều có cờ opt-in mặc định `False`; notebook không tự tải model, clone repo, cài package hay chạy train khi chỉ chạy kiểm kê.

## 0. Environment và mount Google Drive

In [3]:
from pathlib import Path
import os, sys, json, csv, math, re, time, random, subprocess, hashlib, platform
from collections import Counter, defaultdict

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print('Không chạy trong Colab. Hãy mount Drive thủ công và cập nhật đường dẫn trong cấu hình.')

print({'python': sys.version.split()[0], 'platform': platform.platform(), 'colab': IN_COLAB})
try:
    import torch
    print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available(),
          '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
except ImportError:
    print('PyTorch chưa có trong môi trường hiện tại.')


Mounted at /content/drive
{'python': '3.13.16', 'platform': 'Linux-6.6.159+-x86_64-with-glibc2.39', 'colab': True}
PyTorch: 2.14.1+cu130 | CUDA: True | GPU: NVIDIA L4


## 1. Cấu hình trung tâm — sửa đường dẫn tại đây
Các mặc định dưới đây được lấy từ notebook cũ; đó là **đường dẫn dự kiến, chưa xác nhận tồn tại trên Drive**. Candidate list cho phép tái sử dụng thư mục đã chuẩn bị khác. Không đổi `DATASET_DIR` sang thư mục nguồn CCPD nếu chưa có labels YOLO chuẩn.

In [15]:
# Chỉ cấu hình tập trung tại cell này.
from pathlib import Path
DRIVE_PROJECT = Path('/content/drive/MyDrive/ComputerVision/A2')
ORIGINAL_DATASET = DRIVE_PROJECT / 'CCPD_reduced'       # chỉ đọc, dữ liệu gốc
MANIFEST_PATH = ORIGINAL_DATASET / 'manifest.csv'

# ====
# CẤU HÌNH ĐƯỜNG DẪN LOCAL LÀM MẶC ĐỊNH
DRIVE_DATASET_DIR = DRIVE_PROJECT / 'yolo_detection'
LOCAL_DATASET_DIR = Path('/content/dataset/yolo_detection')
DATASET_DIR = LOCAL_DATASET_DIR
COPY_DATASET_TO_LOCAL = False  # Đã tải trực tiếp từ GCS, tắt copy từ Drive

# Chỉ kiểm kê trên Local để tăng tốc
PREPARED_DATASET_CANDIDATES = [
    LOCAL_DATASET_DIR
]
# ====

DRIVE_OUTPUT_ROOT = DRIVE_PROJECT / 'outputs' / 'yolov5s_e0'
OUTPUT_ROOT = Path('/content/outputs/yolov5s_e0')

YOLOV5_DIR = Path('/content/yolov5')
PRETRAINED_WEIGHTS = Path('/content/yolov5s.pt')
KNOWN_OUTPUT_DIRS = [DRIVE_PROJECT/'outputs'/'audit_reports', DRIVE_PROJECT/'outputs'/'yolov5', OUTPUT_ROOT]

CLASS_NAMES = ['license_plate']
NUM_CLASSES = len(CLASS_NAMES)
IMAGE_SIZE = 640
BATCH_SIZE = 32
EPOCHS = 50          # giảm từ 100 để kịp hạn trên Colab; ghi lại giá trị thực tế trong báo cáo
PATIENCE = 15        # early stopping của YOLOv5 (số epoch không cải thiện)
WORKERS = 8
SEED = 42
CONF_THRESHOLD = 0.25
NMS_IOU = 0.45
PAPER_IOU = 0.7      # tiêu chí paper (Tao et al., mục 4.3): box đúng khi IoU > 0.7
EXPECTED_SPLITS = ('train', 'val', 'test')
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

print('Project:', DRIVE_PROJECT)
print('Original (read-only):', ORIGINAL_DATASET)
print('Manifest:', MANIFEST_PATH)
print('Prepared candidates:', *PREPARED_DATASET_CANDIDATES, sep='\n  ')
print('Selected dataset:', DATASET_DIR)
print('Output:', OUTPUT_ROOT)
print('Existing output locations to inspect:', *KNOWN_OUTPUT_DIRS, sep='\n  ')

Project: /content/drive/MyDrive/ComputerVision/A2
Original (read-only): /content/drive/MyDrive/ComputerVision/A2/CCPD_reduced
Manifest: /content/drive/MyDrive/ComputerVision/A2/CCPD_reduced/manifest.csv
Prepared candidates:
  /content/dataset/yolo_detection
Selected dataset: /content/dataset/yolo_detection
Output: /content/outputs/yolov5s_e0
Existing output locations to inspect:
  /content/drive/MyDrive/ComputerVision/A2/outputs/audit_reports
  /content/drive/MyDrive/ComputerVision/A2/outputs/yolov5
  /content/outputs/yolov5s_e0


In [ ]:
# import os
# import shutil
# from pathlib import Path

# # Tìm kiếm các thư mục có cấu trúc YOLO trong thư mục A2
# drive_project = Path('/content/drive/MyDrive/ComputerVision/A2')
# print(f"Đang quét tìm thư mục YOLO dataset trong: {drive_project}...")

# found_datasets = []
# if drive_project.exists():
#     for root, dirs, files in os.walk(drive_project):
#         root_path = Path(root)
#         # Kiểm tra nếu có đủ images/train, labels/train
#         if (root_path / 'images' / 'train').is_dir() and (root_path / 'labels' / 'train').is_dir():
#             found_datasets.append(root_path)
#             # Bỏ qua các thư mục con bên trong thư mục dataset đã tìm thấy
#             dirs[:] = []

# if found_datasets:
#     print("\nĐã tìm thấy các thư mục có cấu trúc YOLO:")
#     for path in found_datasets:
#         print(f"- {path}")
# else:
#     print("\nKhông tìm thấy thư mục nào có cấu trúc YOLO (chứa images/train và labels/train) trong thư mục A2.")

# # --- THỰC HIỆN COPY DATASET VỀ LOCAL ---
# if COPY_DATASET_TO_LOCAL:
#     print(f"\n--- ĐANG COPY DATASET VỀ LOCAL THỰC THI NHANH ---")
#     print(f"Từ: {DRIVE_DATASET_DIR}\nSang: {LOCAL_DATASET_DIR}")
#     LOCAL_DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)
#     if not LOCAL_DATASET_DIR.exists():
#         shutil.copytree(DRIVE_DATASET_DIR, LOCAL_DATASET_DIR)
#         print("Copy hoàn tất!")
#     else:
#         print("Thư mục local đã tồn tại. Bỏ qua bước copy.")

In [1]:
# @title Tải dữ liệu từ Google storage
from google.cloud import storage
from google.colab import auth

auth.authenticate_user()
print("Đã xác thực tài khoản Google Cloud thành công!")

client = storage.Client()
bucket = client.bucket('hoctran-dataset-bucket')

# ---- THAY ĐỔI TÊN DATASET BẠN MUỐN DÙNG Ở ĐÂY ----
target_dataset = 'yolo_detection.zip'

print(f"Đang tải {target_dataset} từ GCS về Colab...")
blob = bucket.blob(target_dataset)
blob.download_to_filename(f'/content/{target_dataset}')

# Giải nén
print("Đang giải nén...")
!unzip -q /content/{target_dataset} -d /content/dataset/
print("Hoàn tất! Sẵn sàng train.")

Đã xác thực tài khoản Google Cloud thành công!
Đang tải yolo_detection.zip từ GCS về Colab...
Đang giải nén...
Hoàn tất! Sẵn sàng train.


## 2. Kiểm kê tài nguyên Drive (read-only)
Cell này chỉ đọc và báo cáo presence, số lượng, split, cấu trúc và file đại diện. Không tự chọn/ghi data, không tạo thư mục. Assets và counts chỉ được xem là đã xác minh sau khi cell chạy trong Colab có mount Drive.

In [16]:
def image_paths(root):
    return sorted(p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS) if root.is_dir() else []

def inventory_dataset(root):
    result = {'path': str(root), 'exists': root.is_dir(), 'splits': {}, 'layout': []}
    if not root.is_dir(): return result
    for split in EXPECTED_SPLITS:
        image_root, label_root = root/'images'/split, root/'labels'/split
        ims = image_paths(image_root)
        lbs = sorted(label_root.rglob('*.txt')) if label_root.is_dir() else []
        result['splits'][split] = {'images':len(ims), 'labels':len(lbs), 'image_dir_exists':image_root.is_dir(), 'label_dir_exists':label_root.is_dir()}
    result['layout'] = [str(p.relative_to(root)) + ('/' if p.is_dir() else '') for p in list(root.iterdir())[:30]]
    return result

manifest_inventory={'exists':MANIFEST_PATH.is_file(),'path':str(MANIFEST_PATH)}
if MANIFEST_PATH.is_file():
    try:
        with MANIFEST_PATH.open(newline='',encoding='utf-8-sig') as f:
            reader=csv.DictReader(f); rows=list(reader)
        manifest_inventory.update({'rows':len(rows),'columns':reader.fieldnames or [],
            'split_counts':dict(Counter(r.get('split','') for r in rows)) if reader.fieldnames and 'split' in reader.fieldnames else None,
            'subset_counts':dict(Counter(r.get('subset','') for r in rows)) if reader.fieldnames and 'subset' in reader.fieldnames else None})
    except Exception as e: manifest_inventory['read_error']=str(e)

inventory = {
  'original_dataset': {'path':str(ORIGINAL_DATASET), 'exists':ORIGINAL_DATASET.is_dir(),
     'top_level': [p.name + ('/' if p.is_dir() else '') for p in list(ORIGINAL_DATASET.iterdir())[:30]] if ORIGINAL_DATASET.is_dir() else [],
     'images_by_split': {s:len(image_paths(ORIGINAL_DATASET/s)) for s in EXPECTED_SPLITS},
     'manifest':manifest_inventory, 'manifest_bytes':MANIFEST_PATH.stat().st_size if MANIFEST_PATH.is_file() else None},
  'prepared_candidates':[inventory_dataset(p) for p in PREPARED_DATASET_CANDIDATES],
  'yolov5_repo':{'path':str(YOLOV5_DIR),'exists':YOLOV5_DIR.is_dir(),'train_py':(YOLOV5_DIR/'train.py').is_file()},
  'pretrained_weights':{'path':str(PRETRAINED_WEIGHTS),'exists':PRETRAINED_WEIGHTS.is_file()},
  'known_scripts':{str(p):p.is_file() for p in [DRIVE_PROJECT/'prepare_yolo_dataset.py', DRIVE_PROJECT/'prepare_dataset.py', DRIVE_PROJECT/'scripts'/'prepare_yolo_dataset.py']},
  'known_output_locations':[{'path':str(root),'exists':root.is_dir(),
      'children':[p.name + ('/' if p.is_dir() else '') for p in list(root.iterdir())[:40]] if root.is_dir() else [],
      'reports_checkpoints_logs':[str(p.relative_to(root)) for p in root.rglob('*') if p.is_file() and (p.suffix.lower() in {'.pt','.json','.csv','.log','.yaml'} or 'report' in p.name.lower())][:100] if root.is_dir() else []} for root in KNOWN_OUTPUT_DIRS]
}
print(json.dumps(inventory, ensure_ascii=False, indent=2))
print('No assets were changed.')


{
  "original_dataset": {
    "path": "/content/drive/MyDrive/ComputerVision/A2/CCPD_reduced",
    "exists": true,
    "top_level": [
      "manifest.csv",
      "validate_ccpd.py",
      ".DS_Store",
      "test/",
      "train/",
      "val/",
      "audit/",
      "reports/",
      "scripts/"
    ],
    "images_by_split": {
      "train": 18000,
      "val": 2000,
      "test": 5000
    },
    "manifest": {
      "exists": true,
      "path": "/content/drive/MyDrive/ComputerVision/A2/CCPD_reduced/manifest.csv",
      "rows": 25000,
      "columns": [
        "original_path",
        "split",
        "subset",
        "filename",
        "output_path"
      ],
      "split_counts": {
        "test": 5000,
        "train": 18000,
        "val": 2000
      },
      "subset_counts": {
        "ccpd_blur": 500,
        "ccpd_challenge": 2500,
        "ccpd_db": 500,
        "ccpd_fn": 500,
        "ccpd_rotate": 500,
        "ccpd_tilt": 500,
        "ccpd_base": 20000
      }
    },
   

## 3. Chọn asset và readiness check
Chọn một prepared dataset thực sự tồn tại và có `images/{train,val,test}` cùng `labels/{train,val,test}`. Cell dừng với hướng dẫn cụ thể nếu chưa đồng bộ/chuẩn bị xong. Không tạo dataset từ CCPD_reduced.

In [17]:
def require(condition, message):
    if not condition: raise RuntimeError(message)

def detect_yolo_layout(root):
    return root.is_dir() and all((root/'images'/s).is_dir() and (root/'labels'/s).is_dir() for s in EXPECTED_SPLITS)

valid_candidates = [p for p in PREPARED_DATASET_CANDIDATES if detect_yolo_layout(p)]
print('Candidates with YOLO split layout:', valid_candidates)
if not detect_yolo_layout(DATASET_DIR) and len(valid_candidates)==1:
    DATASET_DIR=valid_candidates[0]
    print('Reusing the only existing candidate with the required layout:',DATASET_DIR)
elif not detect_yolo_layout(DATASET_DIR) and len(valid_candidates)>1:
    print('Multiple candidates are present. Set DATASET_DIR in the central configuration to the verified dataset.')
if detect_yolo_layout(DATASET_DIR):
    print('Selected dataset layout is present:', DATASET_DIR)
else:
    print('ACTION REQUIRED: dataset is not ready. Synchronize/prepare it later, select its existing path above, then rerun. No copy/rebuild was attempted.')


Candidates with YOLO split layout: [PosixPath('/content/dataset/yolo_detection')]
Selected dataset layout is present: /content/dataset/yolo_detection


## 4. Đối chiếu dataset, annotation và manifest bằng relative path đầy đủ
Kiểm tra ảnh đọc được/kích thước, label rỗng hoặc sai định dạng, class id, normalized xywh và box bounds. Khóa ảnh/label bằng đường dẫn tương đối sau `images/<split>` và `labels/<split>` nên các basename trùng ở subset khác nhau không bị ghép nhầm. Manifest CCPD chỉ dùng đối chiếu nếu schema/path map tương thích; không giả định schema.

**Diễn giải kết quả đã lưu:** `all_valid` chỉ là kiểm tra ảnh/nhãn YOLO; không chứng nhận manifest. Các path manifest và ảnh đã lưu hiện không khớp sau chuẩn hóa cơ bản. Không suy diễn chúng là cùng file nếu thiếu dataset/manifest gốc; chạy lại kiểm tra trên cả hai nguồn để xác minh.

## 4. Dataset synchronization — tạm hoãn

Theo quyết định hiện tại, **không xóa, copy, rebuild hoặc đồng bộ dataset trong notebook này**. Dataset gốc trên Google Drive được giữ nguyên.

Khi bạn đồng bộ dataset sau, hãy cập nhật `DRIVE_DATASET_DIR` ở cell cấu hình trung tâm và chạy lại cell chọn asset/validation. Không chạy các script chuẩn bị dữ liệu cũ trong notebook này.

In [18]:
from PIL import Image

def rel_stem(path, root): return path.relative_to(root).with_suffix('').as_posix()

def validate_yolo_dataset(root, manifest_path=None):
    require(detect_yolo_layout(root), f'Không thấy cấu trúc YOLO đầy đủ ở {root}. Chạy kiểm kê, chọn đúng prepared dataset. Dataset chưa sẵn sàng; không tự dựng.')
    report={'dataset':str(root),'splits':{},'all_valid':True,'all_valid_scope':'image_and_label_integrity_only','overall_validation_status':'incomplete_manifest_path_reconciliation'}
    image_index={}
    for split in EXPECTED_SPLITS:
        ir, lr=root/'images'/split, root/'labels'/split
        ims=image_paths(ir); lbs=sorted(p for p in lr.rglob('*.txt') if p.is_file())
        imap={rel_stem(p,ir):p for p in ims}; lmap={rel_stem(p,lr):p for p in lbs}
        missing=sorted(set(imap)-set(lmap)); orphan=sorted(set(lmap)-set(imap)); invalid=[]; sizes=[]; boxes=0; empty=0
        for key, ip in imap.items():
            try:
                with Image.open(ip) as im: im.verify()
                with Image.open(ip) as im: w,h=im.size
                sizes.append((w,h))
            except Exception as e:
                invalid.append({'path':str(ip.relative_to(root)),'error':'unreadable image: '+str(e)})
        for key, lp in lmap.items():
            try:
                lines=lp.read_text(encoding='utf-8').splitlines()
                if not lines: empty+=1
                for line_no,line in enumerate(lines,1):
                    v=line.split()
                    if len(v)!=5: raise ValueError(f'line {line_no}: expected 5 fields')
                    cls,x,y,bw,bh=map(float,v)
                    if not all(math.isfinite(z) for z in (cls,x,y,bw,bh)): raise ValueError(f'line {line_no}: non-finite value')
                    if cls!=0 or not (0<=x<=1 and 0<=y<=1 and 0<bw<=1 and 0<bh<=1): raise ValueError(f'line {line_no}: class/normalized xywh invalid')
                    if x-bw/2 < -1e-6 or x+bw/2 > 1+1e-6 or y-bh/2 < -1e-6 or y+bh/2 > 1+1e-6: raise ValueError(f'line {line_no}: bbox exceeds image bounds')
                    boxes+=1
            except Exception as e: invalid.append({'path':str(lp.relative_to(root)),'error':str(e)})
        stat={'images':len(ims),'labels':len(lbs),'boxes':boxes,'empty_labels':empty,'missing_labels':len(missing),'orphan_labels':len(orphan),'invalid':len(invalid),
              'image_sizes_unique':len(set(sizes)),'image_size_min':(min((z[0] for z in sizes),default=None),min((z[1] for z in sizes),default=None)),
              'examples':{'missing':missing[:5],'orphan':orphan[:5],'invalid':invalid[:5]}}
        report['splits'][split]=stat
        if missing or orphan or invalid or not ims: report['all_valid']=False
    report['manifest']={'loaded':False,'path_comparison_status':'unverified','reason':'Manifest chưa được nạp hoặc đối chiếu; all_valid chỉ bao gồm tính toàn vẹn ảnh/nhãn.'}
    if manifest_path and Path(manifest_path).is_file():
        try:
            import pandas as pd
            m=pd.read_csv(manifest_path)
            report['manifest'].update({'loaded':True,'rows':len(m),'columns':list(m.columns),'split_counts':m['split'].value_counts(dropna=False).to_dict() if 'split' in m else None,'comparison_method':'output_path minus optional images/<split> or <split> prefix; suffix removed; no subset-folder flattening or basename aliasing'})
            if {'split','output_path'}.issubset(m.columns):
                comparisons={}
                for split in EXPECTED_SPLITS:
                    actual={p.relative_to(root/'images'/split).with_suffix('').as_posix() for p in image_paths(root/'images'/split)}
                    rows=m[m['split'].astype(str)==split]
                    expected=set()
                    for raw in rows['output_path'].astype(str):
                        q=Path(raw)
                        parts=q.parts
                        if len(parts)>=3 and parts[0]=='images' and parts[1]==split: q=Path(*parts[2:])
                        elif len(parts)>=2 and parts[0]==split: q=Path(*parts[1:])
                        expected.add(q.with_suffix('').as_posix())
                    comparisons[split]={'manifest_paths':len(expected),'image_paths':len(actual),'missing_from_images':len(expected-actual),'not_in_manifest':len(actual-expected),'examples_missing':sorted(expected-actual)[:5],'examples_extra':sorted(actual-expected)[:5]}
                report['manifest']['path_comparison']=comparisons
                matched=all(not c['missing_from_images'] and not c['not_in_manifest'] for c in comparisons.values())
                report['manifest']['path_comparison_status']='string_sets_match' if matched else 'string_sets_mismatch_unresolved'
                report['manifest']['file_identity_verified']=False
                if matched: report['overall_validation_status']='path_sets_match_file_identity_unverified'
                if not matched: report['manifest']['reason']='Đã nạp manifest và so sánh path sau chuẩn hóa cơ bản, nhưng các tập path không khớp. Không suy diễn alias thư mục/tên file; cần xác minh trên manifest và dataset gốc.'
                else: report['manifest']['reason']='Các tập path tương đối khớp; điều này chưa xác minh nội dung/định danh file gốc.'
            else: report['manifest']['reason']='Không có cả cột split và output_path; không suy đoán cách ghép.'
        except Exception as e: report['manifest']={'loaded':False,'path_comparison_status':'unverified','error':str(e)}
    return report

if detect_yolo_layout(DATASET_DIR):
    dataset_validation=validate_yolo_dataset(DATASET_DIR, MANIFEST_PATH)
    print(json.dumps(dataset_validation,ensure_ascii=False,indent=2))
    require(dataset_validation['all_valid'], 'Dataset có lỗi image/label. Xem report/examples; sửa dataset bằng quy trình riêng, sau đó chạy lại. Không tự chỉnh dữ liệu ở notebook này.')
else:
    dataset_validation=None
    print('Validation chưa chạy vì dataset chưa sẵn sàng.')


{
  "dataset": "/content/dataset/yolo_detection",
  "splits": {
    "train": {
      "images": 18000,
      "labels": 18000,
      "boxes": 18000,
      "empty_labels": 0,
      "missing_labels": 0,
      "orphan_labels": 0,
      "invalid": 0,
      "image_sizes_unique": 1,
      "image_size_min": [
        720,
        1160
      ],
      "examples": {
        "missing": [],
        "orphan": [],
        "invalid": []
      }
    },
    "val": {
      "images": 2000,
      "labels": 2000,
      "boxes": 2000,
      "empty_labels": 0,
      "missing_labels": 0,
      "orphan_labels": 0,
      "invalid": 0,
      "image_sizes_unique": 1,
      "image_size_min": [
        720,
        1160
      ],
      "examples": {
        "missing": [],
        "orphan": [],
        "invalid": []
      }
    },
    "test": {
      "images": 5000,
      "labels": 5000,
      "boxes": 5000,
      "empty_labels": 0,
      "missing_labels": 0,
      "orphan_labels": 0,
      "invalid": 0,
      "image_s

## 5. EDA và kiểm tra trực quan
Thống kê số ảnh/box, kích thước ảnh và phân bố subset từ đường dẫn tương đối. Vẽ mẫu box từ labels YOLO để xem annotation; chỉ đọc ảnh/label. Kiểm tra license của nguồn dữ liệu cần được ghi vào báo cáo từ điều khoản dataset gốc; notebook không suy đoán license từ tên CCPD.

In [11]:
import matplotlib.pyplot as plt
from PIL import ImageDraw

def box_iou_xyxy(a,b):
    ix1,iy1=max(a[0],b[0]),max(a[1],b[1]); ix2,iy2=min(a[2],b[2]),min(a[3],b[3])
    inter=max(0,ix2-ix1)*max(0,iy2-iy1)
    aa=max(0,a[2]-a[0])*max(0,a[3]-a[1]); ab=max(0,b[2]-b[0])*max(0,b[3]-b[1])
    return inter/max(aa+ab-inter,1e-12)

def subset_of(path, split_root):
    rel=path.relative_to(split_root)
    return rel.parts[0] if len(rel.parts)>1 else '(root)'

def yolo_boxes(label_path,w,h):
    out=[]
    if not label_path.exists(): return out
    for line in label_path.read_text().splitlines():
        if not line.strip(): continue
        _,x,y,bw,bh=map(float,line.split()); out.append(((x-bw/2)*w,(y-bh/2)*h,(x+bw/2)*w,(y+bh/2)*h))
    return out

if dataset_validation:
    for split in EXPECTED_SPLITS:
        ir=DATASET_DIR/'images'/split; ims=image_paths(ir)
        subset_counts=Counter(p.relative_to(ir).parts[0] if len(p.relative_to(ir).parts)>1 else '(root)' for p in ims)
        print(split, dataset_validation['splits'][split], '| subsets:',dict(subset_counts))
        if split=='test' and set(subset_counts)=={'(root)'}:
            print('CẢNH BÁO: images/test không có thư mục con theo subset (DB/FN/Rotate/Tilt/Weather/Challenge...). '
                  'Phân tích theo subset (mục 12, 12b, 13) sẽ chỉ có một nhóm "(root)". '
                  'Nếu cần bảng theo subset như paper, dùng cột subset trong manifest để dựng cấu trúc thư mục ở bước chuẩn bị dữ liệu (ngoài notebook này).')
    sample=[]
    for s in EXPECTED_SPLITS:
        sample += random.Random(SEED).sample(image_paths(DATASET_DIR/'images'/s), min(3,len(image_paths(DATASET_DIR/'images'/s))))
    if sample:
        fig,axes=plt.subplots(len(sample),1,figsize=(12,4*len(sample)))
        axes=[axes] if len(sample)==1 else axes
        for ax,p in zip(axes,sample):
            with Image.open(p) as im: img=im.convert('RGB')
            split=p.relative_to(DATASET_DIR/'images').parts[0]
            lab=DATASET_DIR/'labels'/split/p.relative_to(DATASET_DIR/'images'/split).with_suffix('.txt')
            for b in yolo_boxes(lab,*img.size): ImageDraw.Draw(img).rectangle(b,outline='lime',width=max(2,img.width//400))
            ax.imshow(img); ax.set_title(str(p.relative_to(DATASET_DIR))); ax.axis('off')
        plt.tight_layout(); plt.show()


## 6. Setup official YOLOv5 (opt-in)
Sử dụng repository `ultralytics/yolov5`, scripts `train.py` và `val.py`. Cell setup có thể clone repo, tải pretrained `yolov5s.pt` và cài requirements, nên mặc định tắt. Nếu repo/weights sẵn có, giữ nguyên và dùng lại. Ghi lại commit để reproducibility.

In [12]:
RUN_SETUP = True  # đổi True có chủ đích sau khi kiểm tra dataset/environment
YOLOV5_REPO_URL='https://github.com/ultralytics/yolov5.git'

if RUN_SETUP:
    if not YOLOV5_DIR.exists(): subprocess.run(['git','clone',YOLOV5_REPO_URL,str(YOLOV5_DIR)],check=True)
    require((YOLOV5_DIR/'train.py').is_file() and (YOLOV5_DIR/'val.py').is_file(), f'YOLOv5 repo không hợp lệ: {YOLOV5_DIR}')
    commit=subprocess.check_output(['git','-C',str(YOLOV5_DIR),'rev-parse','HEAD'],text=True).strip()
    if not PRETRAINED_WEIGHTS.is_file():
        import urllib.request
        urllib.request.urlretrieve('https://github.com/ultralytics/yolov5/releases/download/v7.0/yolov5s.pt',PRETRAINED_WEIGHTS)
    subprocess.run([sys.executable,'-m','pip','install','-r',str(YOLOV5_DIR/'requirements.txt')],check=True)
    print('YOLOv5 commit:',commit,'weights:',PRETRAINED_WEIGHTS)
else:
    print('Setup chưa chạy. Repo:',YOLOV5_DIR.exists(),'weights:',PRETRAINED_WEIGHTS.is_file(),'. Bật RUN_SETUP=True để setup nếu cần.')


YOLOv5 commit: cb2f17fd5d790cfdd0c07c25351904df9affbe6e weights: /content/yolov5s.pt


## 7. Tạo data.yaml dùng cấu trúc dataset đã chọn
Chỉ tạo file cấu hình trong `OUTPUT_ROOT`; không sửa dataset. Cần dataset pass validation trước.

In [19]:
RUN_WRITE_CONFIG = True
DATA_YAML = LOCAL_DATASET_DIR/'data.yaml'
if RUN_WRITE_CONFIG:
    require(dataset_validation is not None and dataset_validation['all_valid'],'Dataset validation phải pass trước.')
    import yaml
    OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
    data_cfg={'path':str(DATASET_DIR.resolve()),'train':'images/train','val':'images/val','test':'images/test','nc':NUM_CLASSES,'names':CLASS_NAMES}
    DATA_YAML.write_text(yaml.safe_dump(data_cfg,sort_keys=False),encoding='utf-8')
    print(DATA_YAML.read_text())
else: print('Chưa ghi data.yaml. Bật RUN_WRITE_CONFIG=True khi dataset đã sẵn sàng.')


path: /content/dataset/yolo_detection
train: images/train
val: images/val
test: images/test
nc: 1
names:
- license_plate



## 8. Train YOLOv5s (opt-in)
Chỉ train khi dataset validation pass, official repo và pretrained weights đã xác minh, CUDA sẵn sàng và `RUN_TRAINING=True`. Train/val sử dụng train/val split; test không được truyền vào train.py. `best.pt`, `last.pt`, `results.csv` và log được lưu trên Drive. Cấu hình mặc định là implementation choice, có thể giảm batch/epochs nếu tài nguyên hạn chế; ghi lại thay đổi trong báo cáo.

In [20]:
RUN_TRAINING = True
RUN_NAME='yolov5s_ccpd_e0'
RUN_DIR=OUTPUT_ROOT/'runs'/RUN_NAME
TRAIN_LOG=OUTPUT_ROOT/'logs'/'training.log'

if RUN_TRAINING:
    require(dataset_validation is not None and dataset_validation['all_valid'],'Dataset validation chưa pass.')
    require((YOLOV5_DIR/'train.py').is_file() and PRETRAINED_WEIGHTS.is_file() and DATA_YAML.is_file(),'Repo, pretrained weights hoặc data.yaml thiếu. Hoàn tất setup/config trước.')
    import torch
    require(torch.cuda.is_available(),'CUDA GPU chưa sẵn sàng. Training không bắt đầu.')
    OUTPUT_ROOT.mkdir(parents=True,exist_ok=True); TRAIN_LOG.parent.mkdir(parents=True,exist_ok=True)
    cmd=[sys.executable,str(YOLOV5_DIR/'train.py'),'--img',str(IMAGE_SIZE),'--batch',str(BATCH_SIZE),'--epochs',str(EPOCHS),
         '--data',str(DATA_YAML),'--weights',str(PRETRAINED_WEIGHTS),'--device','0','--workers',str(WORKERS),'--patience',str(PATIENCE),
         '--project',str(RUN_DIR.parent),'--name',RUN_NAME,'--exist-ok','--seed',str(SEED)]
    print('Running:', ' '.join(cmd))
    with TRAIN_LOG.open('w') as log: proc=subprocess.run(cmd,cwd=YOLOV5_DIR,stdout=log,stderr=subprocess.STDOUT,text=True)
    require(proc.returncode==0,f'Training failed; inspect {TRAIN_LOG}')
    print('Training complete:',RUN_DIR,'log:',TRAIN_LOG)
else: print('Training is OFF. No model was trained. Set RUN_TRAINING=True only when ready.')


Running: /usr/bin/python3 /content/yolov5/train.py --img 640 --batch 32 --epochs 50 --data /content/dataset/yolo_detection/data.yaml --weights /content/yolov5s.pt --device 0 --workers 8 --patience 15 --project /content/outputs/yolov5s_e0/runs --name yolov5s_ccpd_e0 --exist-ok --seed 42
Training complete: /content/outputs/yolov5s_e0/runs/yolov5s_ccpd_e0 log: /content/outputs/yolov5s_e0/logs/training.log


## 9. Xác minh checkpoint và logs
Không có checkpoint giả định trước. Cell xác minh file thực tế, kích thước, training results và có thể load checkpoint bằng torch trên CPU.

In [32]:
import sys
from pathlib import Path

BEST_PT=RUN_DIR/'weights'/'best.pt'; LAST_PT=RUN_DIR/'weights'/'last.pt'; RESULTS_CSV=RUN_DIR/'results.csv'
RUN_VERIFY_CHECKPOINT=True

if RUN_VERIFY_CHECKPOINT:
    require(BEST_PT.is_file() and BEST_PT.stat().st_size>0,f'Missing/empty best.pt: {BEST_PT}')
    require(LAST_PT.is_file() and LAST_PT.stat().st_size>0,f'Missing/empty last.pt: {LAST_PT}')
    require(RESULTS_CSV.is_file() and RESULTS_CSV.stat().st_size>0,f'Missing results.csv: {RESULTS_CSV}')

    # Thêm YOLOv5 vào sys.path để tránh lỗi: ModuleNotFoundError: No module named 'models'
    if str(YOLOV5_DIR) not in sys.path:
        sys.path.append(str(YOLOV5_DIR))

    import torch
    ckpt=torch.load(BEST_PT,map_location='cpu',weights_only=False)
    require(isinstance(ckpt,dict) and ('model' in ckpt or 'ema' in ckpt),'Checkpoint structure is unexpected.')
    print({'best_pt_bytes':BEST_PT.stat().st_size,'last_pt_bytes':LAST_PT.stat().st_size,'results_csv_bytes':RESULTS_CSV.stat().st_size,'checkpoint_keys':list(ckpt)})
else: print('Checkpoint verification chưa chạy. Bật RUN_VERIFY_CHECKPOINT=True sau training hoặc khi trỏ RUN_DIR tới run hiện có.')

{'best_pt_bytes': 14358063, 'last_pt_bytes': 14358063, 'results_csv_bytes': 13524, 'checkpoint_keys': ['epoch', 'best_fitness', 'model', 'ema', 'updates', 'optimizer', 'opt', 'git', 'date']}


## 9b. Ghi lại augmentation và siêu tham số thực tế
Đề A2 (mục 7.1.2) yêu cầu trình bày Dataset, DataLoader, augmentation. YOLOv5 đọc augmentation từ file hyp (mặc định `hyp.scratch-low.yaml` với `train.py`) và lưu bản dùng thực tế vào `hyp.yaml`/`opt.yaml` trong thư mục run. Cell này đọc lại các file đó để đưa đúng giá trị vào báo cáo.

Lưu ý: `fliplr` > 0 lật ảnh ngang. Với **detection** biển số điều này chấp nhận được, nhưng với **recognition** (E1/E2) không được lật vì làm đảo ký tự.

In [33]:
RUN_DOC_AUGMENTATION=True
if RUN_DOC_AUGMENTATION:
    import yaml
    hyp_path, opt_path = RUN_DIR/'hyp.yaml', RUN_DIR/'opt.yaml'
    require(hyp_path.is_file(), f'Không thấy {hyp_path}. Chạy training hoặc trỏ RUN_DIR tới run hiện có.')
    hyp=yaml.safe_load(hyp_path.read_text())
    opt=yaml.safe_load(opt_path.read_text()) if opt_path.is_file() else {}
    aug_keys=['hsv_h','hsv_s','hsv_v','degrees','translate','scale','shear','perspective','flipud','fliplr','mosaic','mixup','copy_paste']
    aug={k:hyp.get(k) for k in aug_keys if k in hyp}
    train_cfg={k:opt.get(k) for k in ['epochs','batch_size','imgsz','weights','hyp','patience','seed','workers','cache','rect'] if k in opt}
    print('Augmentation:',json.dumps(aug,indent=2)); print('Train config:',json.dumps(train_cfg,indent=2,default=str))
    if (hyp.get('fliplr') or 0)>0: print('Ghi chú: fliplr>0 — ổn cho detection, KHÔNG dùng cho recognition.')
    OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
    (OUTPUT_ROOT/'augmentation_config.json').write_text(json.dumps({'augmentation':aug,'train_config':train_cfg,'hyp_file':str(hyp_path)},indent=2,default=str))
else: print('Chưa ghi augmentation. Bật RUN_DOC_AUGMENTATION=True sau training.')

Augmentation: {
  "hsv_h": 0.015,
  "hsv_s": 0.7,
  "hsv_v": 0.4,
  "degrees": 0.0,
  "translate": 0.1,
  "scale": 0.5,
  "shear": 0.0,
  "perspective": 0.0,
  "flipud": 0.0,
  "fliplr": 0.5,
  "mosaic": 1.0,
  "mixup": 0.0,
  "copy_paste": 0.0
}
Train config: {
  "epochs": 50,
  "batch_size": 32,
  "imgsz": 640,
  "weights": "/content/yolov5s.pt",
  "hyp": {
    "lr0": 0.01,
    "lrf": 0.01,
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "warmup_epochs": 3.0,
    "warmup_momentum": 0.8,
    "warmup_bias_lr": 0.1,
    "box": 0.05,
    "cls": 0.5,
    "cls_pw": 1.0,
    "obj": 1.0,
    "obj_pw": 1.0,
    "iou_t": 0.2,
    "anchor_t": 4.0,
    "fl_gamma": 0.0,
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
    "degrees": 0.0,
    "translate": 0.1,
    "scale": 0.5,
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.5,
    "mosaic": 1.0,
    "mixup": 0.0,
    "copy_paste": 0.0
  },
  "patience": 15,
  "seed": 42,
  "workers": 8,
  "cache": null,


## 10. Validation evaluation
Validation là split dùng để chọn model/configuration. Không dùng test để chọn threshold hoặc tinh chỉnh. Kết quả stdout và thư mục YOLOv5 được giữ để truy xuất metrics theo phiên bản.

In [34]:
RUN_VALIDATION=True
VAL_DIR=OUTPUT_ROOT/'evaluation'/'validation'; EVAL_LOG_DIR=OUTPUT_ROOT/'logs'
if RUN_VALIDATION:
    require(BEST_PT.is_file() and DATA_YAML.is_file() and (YOLOV5_DIR/'val.py').is_file(),'Cần checkpoint best, data.yaml và val.py.')
    VAL_DIR.mkdir(parents=True,exist_ok=True); EVAL_LOG_DIR.mkdir(parents=True,exist_ok=True)
    cmd=[sys.executable,str(YOLOV5_DIR/'val.py'),'--data',str(DATA_YAML),'--weights',str(BEST_PT),'--img',str(IMAGE_SIZE),
         '--batch',str(BATCH_SIZE),'--device','0' if __import__('torch').cuda.is_available() else 'cpu','--task','val',
         '--project',str(VAL_DIR),'--name','yolov5s_val','--exist-ok']
    val_log=EVAL_LOG_DIR/'validation.log'
    with val_log.open('w') as f: res=subprocess.run(cmd,cwd=YOLOV5_DIR,stdout=f,stderr=subprocess.STDOUT,text=True)
    require(res.returncode==0,f'Validation evaluation failed; inspect {val_log}')
    print(val_log.read_text()[-6000:])
else: print('Validation eval is OFF. Set RUN_VALIDATION=True after verifying checkpoint.')


       P          R      mAP50   mAP50-95: 33% ━━━━──────── 21/63 4.5it/s 6.2s<9.3s
                 Class     Images  Instances          P          R      mAP50   mAP50-95: 34% ━━━━──────── 22/63 4.6it/s 6.5s<9.0s
                 Class     Images  Instances          P          R      mAP50   mAP50-95: 36% ━━━━──────── 23/63 4.6it/s 6.7s<8.8s
                 Class     Images  Instances          P          R      mAP50   mAP50-95: 38% ━━━━╸─────── 24/63 4.5it/s 6.9s<8.7s
                 Class     Images  Instances          P          R      mAP50   mAP50-95: 39% ━━━━╸─────── 25/63 4.5it/s 7.1s<8.4s
                 Class     Images  Instances          P          R      mAP50   mAP50-95: 41% ━━━━╸─────── 26/63 4.5it/s 7.3s<8.2s
                 Class     Images  Instances          P          R      mAP50   mAP50-95: 42% ━━━━━─────── 27/63 4.5it/s 7.6s<7.9s
                 Class     Images  Instances          P          R      mAP50   mAP50-95: 44% ━━━━━─────── 28/63 4.6it/s 7.8s<7.7s

## 11. Final test evaluation — chạy một lần sau khi chốt model
Chạy checkpoint được chọn trên test để báo cáo cuối. Chỉ bật sau khi đã khóa cấu hình dựa trên validation. Không dùng kết quả test để tuning; nếu đã xem test để điều chỉnh, cần ghi rõ hạn chế và tránh mô tả test là holdout độc lập.

In [35]:
RUN_FINAL_TEST=True
TEST_DIR=OUTPUT_ROOT/'evaluation'/'test_final'
if RUN_FINAL_TEST:
    require(BEST_PT.is_file() and DATA_YAML.is_file() and dataset_validation and dataset_validation['all_valid'],'Checkpoint/config/dataset validation chưa sẵn sàng.')
    TEST_DIR.mkdir(parents=True,exist_ok=True); EVAL_LOG_DIR.mkdir(parents=True,exist_ok=True)
    cmd=[sys.executable,str(YOLOV5_DIR/'val.py'),'--data',str(DATA_YAML),'--weights',str(BEST_PT),'--img',str(IMAGE_SIZE),
         '--batch',str(BATCH_SIZE),'--device','0' if __import__('torch').cuda.is_available() else 'cpu','--task','test',
         '--project',str(TEST_DIR),'--name','yolov5s_test_final','--exist-ok']
    test_log=EVAL_LOG_DIR/'test_final.log'
    with test_log.open('w') as f: res=subprocess.run(cmd,cwd=YOLOV5_DIR,stdout=f,stderr=subprocess.STDOUT,text=True)
    require(res.returncode==0,f'Test evaluation failed; inspect {test_log}')
    print(test_log.read_text()[-6000:])
else: print('Final test is OFF. Lock model/config on validation first; then set RUN_FINAL_TEST=True.')


ss     Images  Instances          P          R      mAP50   mAP50-95: 73% ━━━━━━━━╸─── 116/157 4.6it/s 26.4s<8.9s
                 Class     Images  Instances          P          R      mAP50   mAP50-95: 74% ━━━━━━━━╸─── 117/157 4.7it/s 26.6s<8.5s
                 Class     Images  Instances          P          R      mAP50   mAP50-95: 75% ━━━━━━━━━─── 118/157 4.7it/s 26.8s<8.2s
                 Class     Images  Instances          P          R      mAP50   mAP50-95: 75% ━━━━━━━━━─── 119/157 4.7it/s 27.0s<8.0s
                 Class     Images  Instances          P          R      mAP50   mAP50-95: 76% ━━━━━━━━━─── 120/157 4.8it/s 27.3s<7.8s
                 Class     Images  Instances          P          R      mAP50   mAP50-95: 77% ━━━━━━━━━─── 121/157 4.8it/s 27.5s<7.5s
                 Class     Images  Instances          P          R      mAP50   mAP50-95: 77% ━━━━━━━━━─── 122/157 4.9it/s 27.7s<7.2s
                 Class     Images  Instances          P          R      mAP50   mA

## 12. Đánh giá riêng từng test subset (nếu có)
Subset được suy ra từ thư mục con tương đối của `images/test`. Nếu không có thư mục con, sẽ có subset `(root)`. Mỗi YAML/list đặt trong output; dataset không bị sửa. Chỉ chạy sau khi model đã khóa; đây vẫn là phân rã của test, không phải cơ hội tuning.

In [36]:
RUN_SUBSET_TEST=True
SUBSET_DIR=OUTPUT_ROOT/'evaluation'/'test_subsets'
if RUN_SUBSET_TEST:
    require(BEST_PT.is_file() and DATA_YAML.is_file(),'Cần checkpoint và data.yaml.')
    import yaml
    SUBSET_DIR.mkdir(parents=True,exist_ok=True); EVAL_LOG_DIR.mkdir(parents=True,exist_ok=True)
    groups=defaultdict(list); test_root=DATASET_DIR/'images'/'test'
    for p in image_paths(test_root):
        rel=p.relative_to(test_root); subset=rel.parts[0] if len(rel.parts)>1 else '(root)'; groups[subset].append(p)
    subset_results={}
    for subset,paths in sorted(groups.items()):
        safe=re.sub(r'[^A-Za-z0-9_.-]+','_',subset) or 'subset'
        lst=SUBSET_DIR/f'{safe}_images.txt'; lst.write_text('\n'.join(str(p.resolve()) for p in paths)+'\n')
        cfg=SUBSET_DIR/f'{safe}_data.yaml'; cfg.write_text(yaml.safe_dump({'path':str(DATASET_DIR.resolve()),'train':'images/train','val':'images/val','test':str(lst),'nc':NUM_CLASSES,'names':CLASS_NAMES},sort_keys=False))
        log=EVAL_LOG_DIR/f'subset_{safe}.log'; out=SUBSET_DIR/'runs'
        cmd=[sys.executable,str(YOLOV5_DIR/'val.py'),'--data',str(cfg),'--weights',str(BEST_PT),'--img',str(IMAGE_SIZE),'--batch',str(BATCH_SIZE),
             '--device','0' if __import__('torch').cuda.is_available() else 'cpu','--task','test','--project',str(out),'--name',safe,'--exist-ok']
        with log.open('w') as f: res=subprocess.run(cmd,cwd=YOLOV5_DIR,stdout=f,stderr=subprocess.STDOUT,text=True)
        require(res.returncode==0,f'Subset {subset} eval failed; see {log}')
        subset_results[subset]={'images':len(paths),'log':str(log),'output':str(out/safe)}
    (SUBSET_DIR/'subset_runs.json').write_text(json.dumps(subset_results,ensure_ascii=False,indent=2))
    print(json.dumps(subset_results,ensure_ascii=False,indent=2))
else: print('Subset evaluation is OFF. Set RUN_SUBSET_TEST=True after model lock.')


{
  "(root)": {
    "images": 5000,
    "log": "/content/outputs/yolov5s_e0/logs/subset__root_.log",
    "output": "/content/outputs/yolov5s_e0/evaluation/test_subsets/runs/_root_"
  }
}


## 12b. Đánh giá theo tiêu chí của paper (IoU > 0.7) và đo tốc độ
Paper (Tao et al., mục 4.3, Bảng 3) coi box phát hiện là đúng khi IoU > 0.7 và báo **độ chính xác theo ảnh** (mỗi ảnh CCPD có đúng một biển số), không phải mAP. Cell này tính cùng loại số đó (lấy box có confidence cao nhất mỗi ảnh) để đặt cạnh mAP@0.5 của `val.py` mà không so sánh nhầm hai thước đo.

Tốc độ đo trên batch=1, đã khởi động GPU, gồm tiền xử lý + suy luận + NMS, không gồm đọc file. Phần cứng khác paper (RTX 2080Ti), nên chỉ dùng FPS để so sánh nội bộ giữa các biến thể của nhóm, không so trực tiếp với 218.3 FPS của paper. Chạy sau khi đã khóa model; đây vẫn là đánh giá cuối trên test, không dùng để tuning.

In [37]:
RUN_PAPER_COMPARABLE_EVAL=True
PAPER_EVAL_SPLIT='test'
PAPER_EVAL_MAX_PER_SUBSET=None   # đặt số nguyên (vd 500) để chạy thử nhanh; None = toàn bộ
PAPER_EVAL_DIR=OUTPUT_ROOT/'evaluation'/'paper_comparable'
if RUN_PAPER_COMPARABLE_EVAL:
    require(BEST_PT.is_file() and dataset_validation and dataset_validation['all_valid'],'Cần checkpoint best.pt và dataset validation thành công.')
    import torch, numpy as np, pandas as pd
    device='cuda' if torch.cuda.is_available() else 'cpu'
    model=torch.hub.load(str(YOLOV5_DIR),'custom',path=str(BEST_PT),source='local',trust_repo=True)
    model.conf=CONF_THRESHOLD; model.iou=NMS_IOU; model.to(device)
    root=DATASET_DIR/'images'/PAPER_EVAL_SPLIT; lroot=DATASET_DIR/'labels'/PAPER_EVAL_SPLIT
    groups=defaultdict(list)
    for p in image_paths(root): groups[subset_of(p,root)].append(p)
    def load_rgb(p):
        with Image.open(p) as im: return np.array(im.convert('RGB'))
    all_paths=[p for ps in groups.values() for p in ps]
    for p in all_paths[:10]: model(load_rgb(p),size=IMAGE_SIZE)          # warm-up
    if device=='cuda': torch.cuda.synchronize()
    rows=[]
    for subset,paths in sorted(groups.items()):
        if PAPER_EVAL_MAX_PER_SUBSET: paths=random.Random(SEED).sample(paths,min(PAPER_EVAL_MAX_PER_SUBSET,len(paths)))
        for p in paths:
            img=load_rgb(p); h,w=img.shape[:2]
            gt=yolo_boxes(lroot/p.relative_to(root).with_suffix('.txt'),w,h)
            if device=='cuda': torch.cuda.synchronize()
            t0=time.perf_counter(); pred=model(img,size=IMAGE_SIZE).xyxy[0].detach().cpu().tolist()
            if device=='cuda': torch.cuda.synchronize()
            ms=(time.perf_counter()-t0)*1000
            top=max(pred,key=lambda q:q[4]) if pred else None
            iou=max((box_iou_xyxy(top[:4],g) for g in gt),default=0.0) if top else 0.0
            rows.append({'subset':subset,'image':str(p.relative_to(DATASET_DIR)),'has_pred':top is not None,'top1_iou':iou,'correct':iou>PAPER_IOU,'ms':ms})
    df=pd.DataFrame(rows)
    by=df.groupby('subset').agg(images=('correct','size'),acc_iou07=('correct','mean'),mean_top1_iou=('top1_iou','mean'),no_detection=('has_pred',lambda s:int((~s).sum())),fps=('ms',lambda s:1000/s.mean()))
    overall={'images':len(df),'acc_iou07':float(df.correct.mean()),'mean_top1_iou':float(df.top1_iou.mean()),'fps':float(1000/df.ms.mean()),
             'conf':CONF_THRESHOLD,'nms_iou':NMS_IOU,'iou_threshold':PAPER_IOU,'device':device,
             'gpu':torch.cuda.get_device_name(0) if device=='cuda' else None}
    PAPER_EVAL_DIR.mkdir(parents=True,exist_ok=True)
    df.to_csv(PAPER_EVAL_DIR/'per_image.csv',index=False); by.to_csv(PAPER_EVAL_DIR/'by_subset.csv')
    (PAPER_EVAL_DIR/'overall.json').write_text(json.dumps(overall,indent=2))
    print(json.dumps(overall,indent=2)); display(by)
else: print('Paper-comparable eval is OFF. Set RUN_PAPER_COMPARABLE_EVAL=True after model lock.')

YOLOv5 🚀 v7.0-570-gcb2f17fd Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (NVIDIA L4, 22563MiB)

Fusing layers... 
Model summary: 68 layers, 7,012,822 parameters, 0 gradients, 15.8 GFLOPs
Adding AutoShape... 
{
  "images": 5000,
  "acc_iou07": 0.8802,
  "mean_top1_iou": 0.8034493575050948,
  "fps": 96.27354580075901,
  "conf": 0.25,
  "nms_iou": 0.45,
  "iou_threshold": 0.7,
  "device": "cuda",
  "gpu": "NVIDIA L4"
}


,images,acc_iou07,mean_top1_iou,no_detection,fps
subset,,,,,
(root),5000,0.8802,0.803449,11,96.273546


## 13. Prediction visualization và failure analysis
Chọn mẫu test theo subset, vẽ ground-truth/prediction; ghép box một-một greedy theo IoU để tạo scaffold TP/FP/FN/localization. Đây là mẫu chẩn đoán, không phải thống kê full-test. Threshold phải được cố định trước khi xem test. Lưu danh sách prediction, box và ảnh so sánh để nhóm ghi nhận lỗi ánh sáng, mờ, nghiêng, nhỏ, che khuất.

In [39]:
RUN_FAILURE_ANALYSIS=True
ANALYSIS_N_PER_SUBSET=5
ANALYSIS_DIR=OUTPUT_ROOT/'failure_analysis'
if RUN_FAILURE_ANALYSIS:
    require(BEST_PT.is_file() and dataset_validation and dataset_validation['all_valid'],'Cần checkpoint và validation thành công.')
    import torch, numpy as np
    from PIL import ImageDraw
    model=torch.hub.load(str(YOLOV5_DIR),'custom',path=str(BEST_PT),source='local',trust_repo=True)
    model.conf=CONF_THRESHOLD; model.iou=NMS_IOU
    device='cuda' if torch.cuda.is_available() else 'cpu'; model.to(device)
    ANALYSIS_DIR.mkdir(parents=True,exist_ok=True)
    troot=DATASET_DIR/'images'/'test'; groups=defaultdict(list)
    for p in image_paths(troot):
        rel=p.relative_to(troot); groups[rel.parts[0] if len(rel.parts)>1 else '(root)'].append(p)
    def iou(a,b):
        ix1,iy1=max(a[0],b[0]),max(a[1],b[1]); ix2,iy2=min(a[2],b[2]),min(a[3],b[3]); inter=max(0,ix2-ix1)*max(0,iy2-iy1)
        aa=max(0,a[2]-a[0])*max(0,a[3]-a[1]); ab=max(0,b[2]-b[0])*max(0,b[3]-b[1]); return inter/max(aa+ab-inter,1e-12)
    records=[]
    for subset,paths in sorted(groups.items()):
        for p in random.Random(SEED).sample(paths,min(ANALYSIS_N_PER_SUBSET,len(paths))):
            split_rel=p.relative_to(troot); lab=DATASET_DIR/'labels'/'test'/split_rel.with_suffix('.txt')
            with Image.open(p) as im: canvas=im.convert('RGB'); w,h=canvas.size
            gt=yolo_boxes(lab,w,h); pred=model(str(p),size=IMAGE_SIZE).xyxy[0].detach().cpu().tolist()
            pairs=sorted(((iou(g,q[:4]),gi,pi) for gi,g in enumerate(gt) for pi,q in enumerate(pred)),reverse=True)
            mg=set(); mp=set(); tp=0; loc=0
            for score,gi,pi in pairs:
                if gi in mg or pi in mp: continue
                if score>=0.5: mg.add(gi);mp.add(pi);tp+=1
                elif score>=0.1: mg.add(gi);mp.add(pi);loc+=1
            draw=ImageDraw.Draw(canvas)
            for b in gt: draw.rectangle(b,outline='lime',width=max(2,w//500))
            for q in pred: draw.rectangle(q[:4],outline='red',width=max(2,w//500))
            out=ANALYSIS_DIR/f'{re.sub(r"[^A-Za-z0-9_.-]+","_",subset)}__{p.stem}.jpg'; canvas.save(out)
            records.append({'subset':subset,'image':str(p.relative_to(DATASET_DIR)),'visualization':str(out),'gt':len(gt),'pred':len(pred),'tp_iou50':tp,'fp':len(pred)-len(mp),'fn':len(gt)-len(mg),'localization_iou_010_050':loc,'confidence':CONF_THRESHOLD,'nms_iou':NMS_IOU})
    import pandas as pd
    df=pd.DataFrame(records); df.to_csv(ANALYSIS_DIR/'sample_failures.csv',index=False)
    summary={'sample_images':len(df),'tp_iou50':int(df.tp_iou50.sum()) if len(df) else 0,'fp':int(df.fp.sum()) if len(df) else 0,'fn':int(df.fn.sum()) if len(df) else 0,'note':'Diagnostic sample only; inspect examples and annotate qualitative causes.'}
    (ANALYSIS_DIR/'summary.json').write_text(json.dumps(summary,indent=2)); print(summary); display(df.head(20))
else: print('Failure analysis is OFF. Set RUN_FAILURE_ANALYSIS=True after checkpoint verification.')


YOLOv5 🚀 v7.0-570-gcb2f17fd Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (NVIDIA L4, 22563MiB)

Fusing layers... 
Model summary: 68 layers, 7,012,822 parameters, 0 gradients, 15.8 GFLOPs
Adding AutoShape... 
{'sample_images': 5, 'tp_iou50': 5, 'fp': 0, 'fn': 0, 'note': 'Diagnostic sample only; inspect examples and annotate qualitative causes.'}


,subset,image,visualization,gt,pred,...,fp,fn,localization_iou_010_050,confidence,nms_iou
0,(root),images/test/ccpd_challenge__0078-1_0-385&422_5...,/content/outputs/yolov5s_e0/failure_analysis/_...,1,1,...,0,0,0,0.25,0.45
1,(root),images/test/ccpd_blur__0236-2_2-237&461_478&54...,/content/outputs/yolov5s_e0/failure_analysis/_...,1,1,...,0,0,0,0.25,0.45
2,(root),images/test/ccpd_challenge__0229-11_0-294&542_...,/content/outputs/yolov5s_e0/failure_analysis/_...,1,1,...,0,0,0,0.25,0.45
3,(root),images/test/ccpd_challenge__0202-3_0-313&483_4...,/content/outputs/yolov5s_e0/failure_analysis/_...,1,1,...,0,0,0,0.25,0.45
4,(root),images/test/ccpd_challenge__0186-1_0-255&506_4...,/content/outputs/yolov5s_e0/failure_analysis/_...,1,1,...,0,0,0,0.25,0.45


## 14. Xuất metrics/logs/report thực tế
Lưu hash checkpoint, commit YOLOv5, cấu hình, trạng thái validation và đường dẫn artifacts. Metric định lượng được parse từ logs khi có; không điền kết quả mẫu. Lưu ý YOLOv5 `val.py` thường lưu `results.txt`/`results.csv` và stdout tùy phiên bản; cần đọc log gốc nếu schema khác.

In [40]:
RUN_EXPORT_REPORT=True
def parse_val_log(path):
    """Đọc dòng 'all' và dòng Speed của YOLOv5 val.py. Trả None nếu không parse được (không bịa số)."""
    path=Path(path)
    if not path.is_file(): return None
    text=path.read_text(errors='ignore'); out={}
    m=re.search(r'^\s*all\s+(\d+)\s+(\d+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)',text,re.M)
    if m: out.update({'images':int(m[1]),'instances':int(m[2]),'precision':float(m[3]),'recall':float(m[4]),'mAP50':float(m[5]),'mAP50_95':float(m[6])})
    s=re.search(r'Speed:\s*([\d.]+)ms pre-process,\s*([\d.]+)ms inference,\s*([\d.]+)ms NMS',text)
    if s: out.update({'speed_ms_pre':float(s[1]),'speed_ms_inf':float(s[2]),'speed_ms_nms':float(s[3])})
    return out or None
if RUN_EXPORT_REPORT:
    require(OUTPUT_ROOT.exists(),'Output directory chưa tồn tại. Chạy setup/config/train/eval cần thiết trước.')
    repo_commit=None
    if (YOLOV5_DIR/'.git').exists(): repo_commit=subprocess.check_output(['git','-C',str(YOLOV5_DIR),'rev-parse','HEAD'],text=True).strip()
    def sha256_file(path):
        h=hashlib.sha256()
        with Path(path).open('rb') as f:
            for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
        return h.hexdigest()
    artifacts={}
    for k,p in {'best_pt':BEST_PT,'last_pt':LAST_PT,'train_results':RESULTS_CSV,'validation_log':EVAL_LOG_DIR/'validation.log','test_log':EVAL_LOG_DIR/'test_final.log'}.items():
        if p.is_file(): artifacts[k]={'path':str(p),'bytes':p.stat().st_size,'sha256':sha256_file(p) if p.suffix=='.pt' else None}
    metrics={'validation':parse_val_log(EVAL_LOG_DIR/'validation.log'),'test':parse_val_log(EVAL_LOG_DIR/'test_final.log'),
             'test_by_subset':{p.stem.replace('subset_',''):parse_val_log(p) for p in sorted(EVAL_LOG_DIR.glob('subset_*.log'))}}
    pe=PAPER_EVAL_DIR/'overall.json'
    metrics['paper_comparable_iou07']=json.loads(pe.read_text()) if pe.is_file() else None
    ac=OUTPUT_ROOT/'augmentation_config.json'
    report={'experiment':'E0 license-plate detection','model':'YOLOv5s','scope':'detection only','dataset':str(DATASET_DIR),'data_yaml':str(DATA_YAML),
      'class_names':CLASS_NAMES,'image_size':IMAGE_SIZE,'batch_size':BATCH_SIZE,'epochs':EPOCHS,'patience':PATIENCE,'seed':SEED,'confidence_threshold':CONF_THRESHOLD,'nms_iou':NMS_IOU,
      'yolov5_commit':repo_commit,'dataset_validation':dataset_validation,'artifacts':artifacts,'metrics':metrics,
      'augmentation':json.loads(ac.read_text()) if ac.is_file() else None,
      'metrics_note':'Chỉ số được parse từ log thật; giá trị None nghĩa là chưa chạy/không parse được. Test chỉ dùng cuối, không dùng để tuning. mAP@0.5 (val.py) và accuracy IoU>0.7 (paper) là hai thước đo khác nhau.',
      'created_at':time.strftime('%Y-%m-%d %H:%M:%S')}
    report_path=OUTPUT_ROOT/'e0_report.json'; report_path.write_text(json.dumps(report,ensure_ascii=False,indent=2,default=str))
    print('Report:',report_path); print(json.dumps(report,ensure_ascii=False,indent=2,default=str))
else: print('Report export is OFF. Set RUN_EXPORT_REPORT=True after the runs you actually performed.')

Report: /content/outputs/yolov5s_e0/e0_report.json
{
  "experiment": "E0 license-plate detection",
  "model": "YOLOv5s",
  "scope": "detection only",
  "dataset": "/content/dataset/yolo_detection",
  "data_yaml": "/content/dataset/yolo_detection/data.yaml",
  "class_names": [
    "license_plate"
  ],
  "image_size": 640,
  "batch_size": 32,
  "epochs": 50,
  "patience": 15,
  "seed": 42,
  "confidence_threshold": 0.25,
  "nms_iou": 0.45,
  "yolov5_commit": "cb2f17fd5d790cfdd0c07c25351904df9affbe6e",
  "dataset_validation": {
    "dataset": "/content/dataset/yolo_detection",
    "splits": {
      "train": {
        "images": 18000,
        "labels": 18000,
        "boxes": 18000,
        "empty_labels": 0,
        "missing_labels": 0,
        "orphan_labels": 0,
        "invalid": 0,
        "image_sizes_unique": 1,
        "image_size_min": [
          720,
          1160
        ],
        "examples": {
          "missing": [],
          "orphan": [],
          "invalid": []
        }

## 15. Checklist bàn giao E0 và liên hệ paper
- [ ] Ghi lại dataset source, license, số lượng ảnh/annotation thực tế, fixed split và lỗi còn lại.
- [ ] Ghi augmentation/config thực tế (chạy cell 9b, xuất `augmentation_config.json`).
- [ ] Báo cáo P/R, mAP@0.5, mAP@0.5:0.95 (từ `e0_report.json`); thêm accuracy IoU>0.7 và FPS từ cell 12b. Không so trực tiếp mAP với 96.7% của paper, và không so FPS khác phần cứng với 218.3 FPS.
- [ ] Lưu `best.pt`, `last.pt`, results, stdout logs, commit repo và cấu hình.
- [ ] Phân tích lỗi theo hình ảnh/subset; không dùng test để tuning.
- [ ] So sánh ít nhất hai phương pháp/biến thể ở phạm vi toàn A2. E0 YOLOv5s có thể làm baseline detector; E1/E2 có recognition task khác, nên nêu rõ metric/đầu ra và tránh so sánh không cùng tác vụ.
- [ ] Trong paper analysis, phân biệt đóng góp detection (YOLOv5) với recognition (PDLPR: IGFE, multi-head attention, parallel decoder); đối chiếu dataset CCPD, thiết lập và giới hạn về miền biển số Trung Quốc với dữ liệu dự án. Không tuyên bố đã reproduce PDLPR.
- [ ] Đưa code, PDF, GitHub Pages và video demo theo brief.

**Tham chiếu tin cậy được cung cấp:** Assignment 2 CO5085 (brief đính kèm, các mục ở trên); Tao et al., *Sensors* (2024), 24, 2791, [DOI](https://doi.org/10.3390/s24092791), [bài báo MDPI](https://www.mdpi.com/1424-8220/24/9/2791).